In [ ]:
import os

!rm -rf sample_data ai801
!git clone https://github.com/joshbrgs/ai801

%cd ai801
!uv sync --all-groups

In [ ]:
import sys
from pathlib import Path

overrides = {}

sys.path.append(os.path.abspath('src'))

# Inside the repo, so a fresh clone loads the committed agents instead of retraining them.
# The first cell deletes and re-clones the repo, so commit (or download) anything new saved here.
MODEL_DIR = Path("models")
OUTPUT_DIR = Path("output")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ENV_KWARGS = dict(natural=True, num_decks=6, penetration=0.75)

In [ ]:
import gymnasium as gym
import blackjack_rl
from blackjack_rl.play import evaluate
from blackjack_rl.policies import basic_strategy

# Baseline basic Strategy

In [ ]:
env = gym.make(blackjack_rl.ENV_ID, **{**ENV_KWARGS, **overrides})
print(evaluate(env, basic_strategy))

# Q Learning

In [ ]:
import copy
from collections import defaultdict

import numpy as np


def state_key(observation):
    return tuple(int(value) for value in observation[:5])

def state_key_with_count(observation):
    """The hand plus the Hi-Lo count bucket the wrapper adds as a 6th value."""
    return tuple(int(value) for value in observation[:6])

def choose_q_action(q_table, observation, info, epsilon, rng, key_fn=state_key):
    legal_actions = np.flatnonzero(info['action_mask'])
    if legal_actions.size == 0:
        raise ValueError('No legal actions were supplied for a non-terminal state.')

    values = q_table.get(key_fn(observation))
    if values is None or rng.random() < epsilon:
        return int(rng.choice(legal_actions))

    legal_values = values[legal_actions]
    best_actions = legal_actions[legal_values == np.max(legal_values)]
    return int(rng.choice(best_actions))


def train_q_learning(env, episodes=300_000, gamma=1.0, epsilon_start=1.0, epsilon_min=0.05,
                     seed=42, key_fn=state_key, checkpoints=(), checkpoint_rounds=100_000):
    """Returns the Q-table and a learning curve of (rounds trained, score of the greedy policy)."""
    if episodes <= 0:
        raise ValueError('episodes must be greater than zero')
    if not 0 <= gamma <= 1:
        raise ValueError('gamma must be in [0, 1]')
    if not 0 <= epsilon_min <= epsilon_start <= 1:
        raise ValueError('epsilon values must satisfy 0 <= epsilon_min <= epsilon_start <= 1')

    q_table = defaultdict(lambda: np.zeros(env.action_space.n, dtype=np.float64))
    visits = defaultdict(lambda: np.zeros(env.action_space.n, dtype=np.float64))
    rng = np.random.default_rng(seed)
    decay_episodes = max(1, int(0.8 * episodes))
    checkpoint_env = copy.deepcopy(env)  # scoring uses its own shoe, so training's shoe isn't disturbed
    curve = []

    for episode in range(episodes):
        progress = min(episode / decay_episodes, 1.0)
        epsilon = epsilon_start + progress * (epsilon_min - epsilon_start)
        observation, info = env.reset(seed=seed if episode == 0 else None)
        terminated = truncated = False

        while not (terminated or truncated):
            state = key_fn(observation)
            action = choose_q_action(q_table, observation, info, epsilon, rng, key_fn)
            next_observation, reward, terminated, truncated, next_info = env.step(action)
            done = terminated or truncated

            target = float(reward)
            if not done:
                next_state = key_fn(next_observation)
                next_legal = np.flatnonzero(next_info['action_mask'])
                target += gamma * np.max(q_table[next_state][next_legal])

            visits[state][action] += 1
            alpha = 1 / visits[state][action] ** 0.75  # big steps at first, smaller with more visits
            q_table[state][action] += alpha * (target - q_table[state][action])
            observation, info = next_observation, next_info

        if episode + 1 in checkpoints:
            score = evaluate(checkpoint_env, make_q_policy(q_table, key_fn), n=checkpoint_rounds)
            curve.append((episode + 1, score))
            print(f'{episode + 1:>9,} rounds: {score}')

    return dict(q_table), curve


def make_q_policy(q_table, key_fn=state_key):
    rng = np.random.default_rng(0)

    def policy(observation, info):
        return choose_q_action(q_table, observation, info, epsilon=0.0, rng=rng, key_fn=key_fn)

    return policy

q_table, _ = train_q_learning(env, episodes=300_000, seed=42)
q_policy = make_q_policy(q_table)
print(f'Learned {len(q_table):,} state-action rows')
print('Greedy policy evaluation:', evaluate(env, q_policy, n=100_000, seed=123))

# Hi-Lo Card Counting

Wraps the game so the agent can also see a Hi-Lo card count. A 6th number tells the agent whether the cards left are good or bad for it.
 - **+1** = many low cards have already been played, so the cards left have more tens and aces (which tends to be good for the player)
 - **0** = close to neutral
 - **-1** = many high cards have already been played (which tends to be worse for the player)

The wrapper is `HiLoCountWrapper` in `src/blackjack_rl/counting.py`, so the pygame table can use it too.

In [ ]:
from blackjack_rl.counting import HiLoCountWrapper

# Create the counting version of the game (the count agent trains on this later).
# Quick check, the last part should print Discrete(3, start=-1)
count_env = HiLoCountWrapper(gym.make(blackjack_rl.ENV_ID, natural=True, num_decks=6))
print(count_env.observation_space)

# Q-learning vs DQN with Hi-Lo counting

In [ ]:
import pickle

import matplotlib.pyplot as plt
import pandas as pd

from blackjack_rl import ACTION_NAMES, DOUBLE, HIT, STICK

TRAIN_ROUNDS = 3_000_000               # training rounds per agent; use 50_000 for a quick check
# Learning-curve points, spaced out on a log scale because most of the learning happens early
CHECKPOINTS = [10_000, 30_000, 100_000, 300_000, 1_000_000, 3_000_000]
CHECKPOINT_ROUNDS = 100_000            # rounds scored at each learning-curve point
EVAL_ROUNDS = 2_000_000                # rounds each agent is scored on in the results table
EVAL_SEED = 2026
FINE_BUCKETS = dict(min_bucket=-3, max_bucket=5)  # 9 count buckets instead of the default 3


def make_count_env(min_bucket=-1, max_bucket=1):
    env = gym.make(blackjack_rl.ENV_ID, **ENV_KWARGS)
    return HiLoCountWrapper(env, min_bucket=min_bucket, max_bucket=max_bucket)


def load_or_train(name, train):
    """Load a saved agent if this training budget was already run, otherwise train and save it."""
    path = MODEL_DIR / f'{name}_{TRAIN_ROUNDS}.pkl'
    if path.exists():
        with open(path, 'rb') as f:
            return pickle.load(f)
    agent = train()
    with open(path, 'wb') as f:
        pickle.dump(agent, f)
    return agent

## Betting

Payouts scale with the bet and there is no bankroll, so the best play for a hand is the same
whether 1 or 8 units are on it. So the agents **train with flat bets**, and betting is only applied
when they are scored: each round is played at 1 unit, and we also record the bet the Hi-Lo spread
would have placed, using the true count *before* the deal (`info['true_count_before_deal']`).

The spread bets 1 unit until the true count reaches +2, then one unit per true count, up to 8.

Because both bet rules are scored on the same rounds, **gain from spread** (`(bet - 1) × reward`)
is a paired comparison with a much smaller error bar than subtracting two separate results.

In [ ]:
import math


def hilo_bet(true_count):
    """1 unit until the true count reaches +2, then one unit per true count, up to 8."""
    return min(max(math.floor(true_count), 1), 8)


def play_rounds(env, policy, n=EVAL_ROUNDS, seed=EVAL_SEED):
    """Play n rounds at 1 unit. Returns each round's reward and the bet the spread would have made."""
    rewards, bets = np.zeros(n), np.zeros(n)
    env.reset(seed=seed)
    for i in range(n):
        observation, info = env.reset()
        bets[i] = hilo_bet(info['true_count_before_deal'])
        terminated = truncated = False
        while not (terminated or truncated):
            observation, reward, terminated, truncated, info = env.step(policy(observation, info))
        rewards[i] = reward
    return rewards, bets


def mean_and_error(values):
    """Mean with its standard error, e.g. '+0.0031 ± 0.0013'."""
    return f'{values.mean():+.4f} ± {values.std() / np.sqrt(len(values)):.4f}'


# Sanity check: flat bets should be near the baseline above (about -0.002), and the average
# spread bet should be clearly above 1 (about 1.3). At this many rounds the gain is still noisy.
basic_rewards, basic_bets = play_rounds(make_count_env(), basic_strategy, n=200_000)
print('flat bets:   ', mean_and_error(basic_rewards))
print('Hi-Lo spread:', mean_and_error(basic_bets * basic_rewards))
print('average bet: ', round(basic_bets.mean(), 2))

# Training Q-Learning agents with counting

In [ ]:
def train_table(key_fn, **buckets):
    return train_q_learning(make_count_env(**buckets), episodes=TRAIN_ROUNDS, key_fn=key_fn,
                            checkpoints=CHECKPOINTS, checkpoint_rounds=CHECKPOINT_ROUNDS)


ql_no_count_table, ql_no_count_curve = load_or_train(
    'ql_no_count', lambda: train_table(state_key))
ql_bucket_table, ql_bucket_curve = load_or_train(
    'ql_3_buckets', lambda: train_table(state_key_with_count))
ql_fine_table, ql_fine_curve = load_or_train(
    'ql_9_buckets', lambda: train_table(state_key_with_count, **FINE_BUCKETS))

ql_no_count_policy = make_q_policy(ql_no_count_table, key_fn=state_key)
ql_bucket_policy = make_q_policy(ql_bucket_table, key_fn=state_key_with_count)
ql_fine_policy = make_q_policy(ql_fine_table, key_fn=state_key_with_count)
print(f'Q-table rows: {len(ql_no_count_table):,} without the count, '
      f'{len(ql_bucket_table):,} with 3 buckets, {len(ql_fine_table):,} with 9 buckets')

# Deep Q-Learning

The DQN follows the same loop as `train_q_learning`, with the Q-table replaced by a small neural
network. Because a network can take a continuous input, it sees the **raw true count** from
`info['true_count']` instead of a bucket.

Three things are new compared with the Q-table:

- **Encoding.** The observation becomes a list of 46 numbers (`encode` in
  `src/blackjack_rl/agents.py`, shared with the pygame table). Player total and dealer card are
  one-hot (one slot each, set to 1), because value doesn't change smoothly with the total: 12 and 16
  play very differently from 17. The true count is clipped to ±6 and scaled to ±1.
- **Replay buffer.** Each move is stored, and the network trains on random batches of past moves
  instead of only the latest one, so it doesn't overfit to the current round.
- **Target network.** The Q-learning target `reward + max Q(next state)` uses a copy of the network
  that is only updated every few thousand steps, so the target doesn't move every time the network
  learns.

Illegal actions are masked out when choosing an action and when computing the target, just like
`np.flatnonzero(info['action_mask'])` does for the table. The learning rate shrinks over training
for the same reason the Q-table's step size does.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from blackjack_rl.agents import encode

torch.set_num_threads(1)  # a network this small runs faster on one thread


def make_q_network():
    """46 inputs -> one Q-value for each of the 4 actions."""
    return nn.Sequential(
        nn.Linear(46, 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 4),
    )

In [ ]:
def choose_dqn_action(q_network, observation, info, epsilon, rng):
    legal_actions = np.flatnonzero(info['action_mask'])
    if rng.random() < epsilon:
        return int(rng.choice(legal_actions))
    with torch.no_grad():
        q_values = q_network(torch.from_numpy(encode(observation, info))).numpy()
    return int(legal_actions[np.argmax(q_values[legal_actions])])


def dqn_train_step(q_network, target_network, optimizer, batch, gamma):
    """One gradient step on a batch of stored moves."""
    states, actions, rewards, next_states, next_masks, finished = zip(*batch)
    states = torch.tensor(np.array(states))
    actions = torch.tensor(actions)
    rewards = torch.tensor(rewards, dtype=torch.float32)
    next_states = torch.tensor(np.array(next_states))
    next_masks = torch.tensor(np.array(next_masks))
    finished = torch.tensor(finished, dtype=torch.float32)

    with torch.no_grad():
        # Same target as the Q-table: reward + gamma * best legal Q-value of the next state
        next_q = target_network(next_states).masked_fill(next_masks == 0, -1e9).max(dim=1).values
        targets = rewards + gamma * next_q * (1 - finished)

    q_values = q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    loss = F.mse_loss(q_values, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


def train_dqn(env, episodes=300_000, gamma=1.0, epsilon_start=1.0, epsilon_min=0.05, seed=42,
              learning_rate=3e-4, batch_size=256, buffer_size=200_000, train_every=4,
              target_update_every=2_000, checkpoints=(), checkpoint_rounds=100_000):
    """Returns the network and a learning curve of (rounds trained, score of the greedy policy)."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    q_network = make_q_network()
    target_network = make_q_network()
    target_network.load_state_dict(q_network.state_dict())
    optimizer = torch.optim.Adam(q_network.parameters(), lr=learning_rate)
    replay_buffer = []
    decay_episodes = max(1, int(0.8 * episodes))
    checkpoint_env = copy.deepcopy(env)  # scoring uses its own shoe, so training's shoe isn't disturbed
    curve = []
    steps = 0

    for episode in range(episodes):
        progress = min(episode / decay_episodes, 1.0)
        epsilon = epsilon_start + progress * (epsilon_min - epsilon_start)
        for group in optimizer.param_groups:  # smaller steps as training goes on, like alpha for the Q-table
            group['lr'] = learning_rate * max(1 - episode / episodes, 0.02)
        observation, info = env.reset(seed=seed if episode == 0 else None)
        terminated = truncated = False

        while not (terminated or truncated):
            action = choose_dqn_action(q_network, observation, info, epsilon, rng)
            next_observation, reward, terminated, truncated, next_info = env.step(action)

            move = (encode(observation, info), action, reward,
                    encode(next_observation, next_info), next_info['action_mask'], terminated)
            if len(replay_buffer) < buffer_size:
                replay_buffer.append(move)
            else:
                replay_buffer[steps % buffer_size] = move  # full: overwrite the oldest moves
            steps += 1

            if len(replay_buffer) >= batch_size and steps % train_every == 0:
                batch = [replay_buffer[i] for i in rng.integers(len(replay_buffer), size=batch_size)]
                dqn_train_step(q_network, target_network, optimizer, batch, gamma)
            if steps % target_update_every == 0:
                target_network.load_state_dict(q_network.state_dict())

            observation, info = next_observation, next_info

        if episode + 1 in checkpoints:
            score = evaluate(checkpoint_env, make_dqn_policy(q_network), n=checkpoint_rounds)
            curve.append((episode + 1, score))
            print(f'{episode + 1:>9,} rounds: {score}')

    return q_network, curve


def make_dqn_policy(q_network):
    rng = np.random.default_rng(0)

    def policy(observation, info):
        return choose_dqn_action(q_network, observation, info, epsilon=0.0, rng=rng)

    return policy

In [ ]:
# DQN training is the slow part, so the trained network is saved and reloaded on reruns
dqn_network, dqn_curve = load_or_train('dqn', lambda: train_dqn(
    make_count_env(), episodes=TRAIN_ROUNDS,
    checkpoints=CHECKPOINTS, checkpoint_rounds=CHECKPOINT_ROUNDS))
dqn_policy = make_dqn_policy(dqn_network)

In [ ]:
basic_score = evaluate(make_count_env(), basic_strategy, n=CHECKPOINT_ROUNDS)

fig, ax = plt.subplots(figsize=(8, 5))
for label, curve in [('Q-learning, no count', ql_no_count_curve),
                     ('Q-learning + 3 count buckets', ql_bucket_curve),
                     ('Q-learning + 9 count buckets', ql_fine_curve),
                     ('DQN + true count', dqn_curve)]:
    rounds = [r for r, _ in curve]
    ax.errorbar(rounds, [s.mean for _, s in curve], yerr=[s.stderr for _, s in curve],
                marker='o', capsize=3, label=label)
ax.axhline(basic_score.mean, linestyle='--', color='gray', label='basic strategy')
ax.set_xscale('log')
ax.set_xlabel('training rounds (log scale)')
ax.set_ylabel('profit per round (flat bets, greedy policy)')
ax.set_title('Learning curves')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'learning_curves.png')
plt.show()

In [ ]:
# Each policy with the bucket settings it was trained on
POLICIES = {
    'Basic strategy': (basic_strategy, {}),
    'Q-learning, no count': (ql_no_count_policy, {}),
    'Q-learning + 3 count buckets': (ql_bucket_policy, {}),
    'Q-learning + 9 count buckets': (ql_fine_policy, FINE_BUCKETS),
    'DQN + true count': (dqn_policy, {}),
}

rows = []
for name, (policy, buckets) in POLICIES.items():
    rewards, bets = play_rounds(make_count_env(**buckets), policy)
    rows.append({
        'policy': name,
        'flat bets': mean_and_error(rewards),
        'Hi-Lo spread': mean_and_error(bets * rewards),
        'gain from spread': mean_and_error((bets - 1) * rewards),
        'average bet': round(bets.mean(), 2),
        'spread profit per unit bet': f'{(bets * rewards).sum() / bets.sum():+.4f}',
    })

results = pd.DataFrame(rows).set_index('policy')
results.to_csv(OUTPUT_DIR / 'results.csv')
display(results)